# Figure 1 — pages
Assembles the panels drawn by `fibroblast_figures.ipynb` (`figures/fig1*.svg`) into `figures/Figure1.{svg,pdf,png}` and
`figures/Figure1S.{svg,pdf,png}`, cropped tight to the content. Rule: rows fill the page width, and each panel's main text
prints at ≥ 7 pt where the layout allows (scale = 7 / the panel's main font size as drawn).

In [1]:
import copy
import io
import xml.etree.ElementTree as ET

import cairosvg
import numpy as np
from PIL import Image

FIG = "figures"
SVG_NS = "http://www.w3.org/2000/svg"
for prefix, uri in [("", SVG_NS), ("xlink", "http://www.w3.org/1999/xlink"),
                    ("rdf", "http://www.w3.org/1999/02/22-rdf-syntax-ns#"), ("cc", "http://creativecommons.org/ns#"),
                    ("dc", "http://purl.org/dc/elements/1.1/")]:
    ET.register_namespace(prefix, uri)                 # keep matplotlib's prefixes when re-serializing
PAGE_W, PAGE_H = 595.276, 841.89                       # A4 (pt)
CONTENT_X, CONTENT_W = 28, 543                         # panel area left edge / width


def load_svg(name):
    """Root element of a panel SVG in figures/."""
    return ET.parse(f"{FIG}/{name}.svg").getroot()


def size(root):
    """(width, height) of an SVG from its viewBox."""
    return tuple(float(v) for v in root.get("viewBox").split()[2:])


def new_page():
    """Blank A4 page with a white background."""
    page = ET.Element(f"{{{SVG_NS}}}svg", width=f"{PAGE_W}pt", height=f"{PAGE_H}pt", viewBox=f"0 0 {PAGE_W} {PAGE_H}")
    ET.SubElement(page, f"{{{SVG_NS}}}rect", x="0", y="0", width=f"{PAGE_W}", height=f"{PAGE_H}", fill="#ffffff")
    return page


def nest(page, src_root, x, y, scale):
    """Place a panel SVG on the page at (x, y), scaled. Returns its bottom."""
    vb = tuple(float(v) for v in src_root.get("viewBox").split())
    child = ET.SubElement(page, f"{{{SVG_NS}}}svg", x=f"{x}", y=f"{y}", width=f"{vb[2] * scale}",
                          height=f"{vb[3] * scale}", viewBox=" ".join(f"{v}" for v in vb),
                          preserveAspectRatio="xMinYMin meet", overflow="hidden")
    for el in src_root:
        child.append(copy.deepcopy(el))
    return y + vb[3] * scale


def letter(page, text, x, y):
    """Panel letter (Arial 12, lowercase + period)."""
    t = ET.SubElement(page, f"{{{SVG_NS}}}text", x=f"{x}", y=f"{y}",
                      style="font-family: Arial, ArialMT, sans-serif; font-size: 12px; fill: #000000")
    t.text = f"{text}."


def write(page, name, pad=4):
    """Crop the page to its drawn content (+ pad pt) and save it as SVG, PDF and PNG in figures/."""
    png = cairosvg.svg2png(bytestring=ET.tostring(page), scale=4)
    ink = np.asarray(Image.open(io.BytesIO(png)).convert("L")) < 250          # drawn (non-white) pixels
    px = ink.shape[1] / PAGE_W                                   # pixels per pt
    rows = np.where(ink.any(axis=1))[0]
    cols = np.where(ink.any(axis=0))[0]
    x0, x1 = max(cols[0] / px - pad, 0), min((cols[-1] + 1) / px + pad, PAGE_W)
    y0, y1 = max(rows[0] / px - pad, 0), min((rows[-1] + 1) / px + pad, PAGE_H)
    page.set("viewBox", f"{x0:.2f} {y0:.2f} {x1 - x0:.2f} {y1 - y0:.2f}")
    page.set("width", f"{x1 - x0:.2f}pt")
    page.set("height", f"{y1 - y0:.2f}pt")
    bg = page.find(f"{{{SVG_NS}}}rect")                          # white background follows the crop
    bg.attrib.update(x=f"{x0:.2f}", y=f"{y0:.2f}", width=f"{x1 - x0:.2f}", height=f"{y1 - y0:.2f}")
    ET.ElementTree(page).write(f"{FIG}/{name}.svg", xml_declaration=True, encoding="utf-8")
    cairosvg.svg2pdf(url=f"{FIG}/{name}.svg", write_to=f"{FIG}/{name}.pdf")
    cairosvg.svg2png(url=f"{FIG}/{name}.svg", write_to=f"{FIG}/{name}.png", scale=2)

## Figure 1
a = Hi-C → LAS → age-specific map (top, full width); b colocalized pathways | c colocalization vs expression (donors);
d interferon-γ score vs age in the population cohort | e IFN-γ core (cited last in the text).

In [2]:
MAIN_FONT = {"fig1b_colocalization": 12, "fig1c_concordance_donors": 10,      # main font size as drawn (pt)
             "fig1d_cohort_ifng_age": 10, "fig1e_ifng_core": 10}
GAP, GAP_X = 12, 18                                    # vertical gap between rows, horizontal gap between panels (pt)
page = new_page()
a_svg = load_svg("fig1a_hic_processing")
bottom = nest(page, a_svg, CONTENT_X, 17, CONTENT_W / size(a_svg)[0])    # full width; its fonts are drawn x1.2 (~7 pt printed)
letter(page, "a", CONTENT_X - 11, 25)

# row 3 first (it sets the remaining budget): e (core) at 7 pt on the right; d (cohort, drawn wide) fills the rest of the
# width at e's height
d_svg = load_svg("fig1d_cohort_ifng_age")
e_svg = load_svg("fig1e_ifng_core")
d_w, d_h = size(d_svg)
e_w, e_h = size(e_svg)
e_s = 7 / MAIN_FONT["fig1e_ifng_core"]
d_s = min((CONTENT_W - GAP_X - e_w * e_s) / d_w, e_h * e_s / d_h)
row3_h = max(d_h * d_s, e_h * e_s)

# row 2: b | c at one shared height filling the width, capped by the height left above row 3
b_svg = load_svg("fig1b_colocalization")
c_svg = load_svg("fig1c_concordance_donors")
b_w, b_h = size(b_svg)
c_w, c_h = size(c_svg)
fill_h = (CONTENT_W - GAP_X) / (b_w / b_h + c_w / c_h)
row2_h = min(fill_h, PAGE_H - 14 - bottom - 2 * GAP - row3_h)
b_s = row2_h / b_h
c_s = row2_h / c_h
y2 = bottom + GAP
nest(page, b_svg, CONTENT_X, y2, b_s)
c_x = CONTENT_X + CONTENT_W - c_w * c_s                    # right-aligned; any slack falls between b and c
nest(page, c_svg, c_x, y2, c_s)
y3 = y2 + row2_h + GAP
nest(page, d_svg, CONTENT_X, y3, d_s)
e_x = CONTENT_X + CONTENT_W - e_w * e_s
end = max(nest(page, e_svg, e_x, y3, e_s), y3 + row3_h)
for lt, x, y in (("b", CONTENT_X, y2), ("c", c_x, y2), ("d", CONTENT_X, y3), ("e", e_x, y3)):
    letter(page, lt, x - (11 if x == CONTENT_X else 4), y + 8)
write(page, "Figure1")

fonts = {"b": b_s * MAIN_FONT["fig1b_colocalization"], "c": c_s * MAIN_FONT["fig1c_concordance_donors"],
         "d": d_s * MAIN_FONT["fig1d_cohort_ifng_age"], "e": e_s * MAIN_FONT["fig1e_ifng_core"]}
print("Figure1: printed main font (pt):", {k: round(v, 1) for k, v in fonts.items()},
      f"| row 2 {row2_h:.0f} pt (fill {fill_h:.0f}), row 3 {row3_h:.0f} pt; bottom at {end:.0f} of {PAGE_H:.0f} pt")

Figure1: printed main font (pt): {'b': 7.6, 'c': 7.3, 'd': 6.9, 'e': 7.0} | row 2 280 pt (fill 287), row 3 249 pt; bottom at 828 of 842 pt


## Supplementary Figure 1S
Two columns, then a full-width row (letters read across rows): left, locus-matched test (a) and cohort concordance (c);
right, Hi-C donor expression (b) and population-cohort expression (d); bottom, young-specific cores, senescence
inhibitors (e) | ECM (f). Each panel at 7 pt / its main font size; one shared shrink factor only if the columns would not fit.

In [3]:
MAIN_FONT_S = {"fig1S_a_locus_matched": 10,           # main font size as drawn (pt)
               "fig1S_c_concordance_cohort": 10,
               "fig1S_b_expression_donors": 12,
               "fig1S_d_expression_cohort": 12}
CORE_MIN, GAP_S, TOP = 200, 10, 12                     # min core-row height, gap between panels, top margin (pt)
page = new_page()
left = [("a", "fig1S_a_locus_matched"), ("c", "fig1S_c_concordance_cohort")]
right = [("b", "fig1S_b_expression_donors"), ("d", "fig1S_d_expression_cohort")]
dims = {n: size(load_svg(n)) for _, n in left + right}
s7 = {n: 7 / MAIN_FONT_S[n] for n in dims}             # scale that prints each panel's main text at 7 pt
w_l = max(dims[n][0] * s7[n] for _, n in left)
w_r = max(dims[n][0] * s7[n] for _, n in right)
h_col = max(sum(dims[n][1] * s7[n] for _, n in col) + GAP_S for col in (left, right))
fit = min(1.0, (CONTENT_W - 14) / (w_l + w_r), (PAGE_H - 14 - TOP - GAP_S - CORE_MIN) / h_col)
right_x = CONTENT_X + CONTENT_W - w_r * fit
# c sits as far right as it can without reaching d's labels, to line its plot up with a's as closely as possible
c_name = "fig1S_c_concordance_cohort"
c_x = right_x - 6 - dims[c_name][0] * s7[c_name] * fit
bottom = TOP
for x0, col in ((CONTENT_X, left), (right_x, right)):
    y = TOP
    for lt, name in col:
        x = c_x if name == c_name else x0
        top = y
        y = nest(page, load_svg(name), x, y, s7[name] * fit) + GAP_S
        letter(page, lt, x0 - (11 if x0 == CONTENT_X else 4), top + 8)   # letters stay on the column edge
    bottom = max(bottom, y)

# bottom row: cores at one shared height, centred
y = bottom
e_svg = load_svg("fig1S_e_seninh_core")
f_svg = load_svg("fig1S_f_ecm_core")
e_w, e_h = size(e_svg)
f_w, f_h = size(f_svg)
height = min((CONTENT_W - 14) / (e_w / e_h + f_w / f_h), PAGE_H - 14 - y)
row_w = e_w * height / e_h + 14 + f_w * height / f_h
e_x = CONTENT_X + (CONTENT_W - row_w) / 2
f_x = e_x + e_w * height / e_h + 14
end = max(nest(page, e_svg, e_x, y, height / e_h), nest(page, f_svg, f_x, y, height / f_h))
letter(page, "e", CONTENT_X - 11, y + 8)
letter(page, "f", f_x - 4, y + 8)
write(page, "Figure1S")
print(f"Figure1S: panels a-d at 7 pt x {fit:.2f} (printed ~{7 * fit:.1f} pt); cores height {height:.0f} pt; "
      f"bottom at {end:.0f} of {PAGE_H:.0f} pt")

Figure1S: panels a-d at 7 pt x 0.95 (printed ~6.6 pt); cores height 266 pt; bottom at 828 of 842 pt
